In [ ]:
import pandas as pd
import xarray as xr
import glob
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np
import os
import datetime as dt
import subprocess
from tropycal import tracks

In [ ]:
from datetime import datetime as dt

track_data = {
    "datetime": [
        "2014-09-10 18:00", "2014-09-11 00:00", "2014-09-11 06:00", "2014-09-11 12:00",
        "2014-09-11 18:00", "2014-09-12 00:00", "2014-09-12 06:00", "2014-09-12 12:00",
        "2014-09-12 18:00", "2014-09-13 00:00", "2014-09-13 06:00", "2014-09-13 12:00",
        "2014-09-13 18:00", "2014-09-14 00:00", "2014-09-14 06:00", "2014-09-14 12:00",
        "2014-09-14 18:00", "2014-09-15 00:00", "2014-09-15 06:00", "2014-09-15 12:00",
        "2014-09-15 18:00", "2014-09-16 00:00", "2014-09-16 06:00", "2014-09-16 12:00",
        "2014-09-16 18:00", "2014-09-17 00:00", "2014-09-17 06:00", "2014-09-17 12:00",
        "2014-09-17 18:00", "2014-09-18 00:00", "2014-09-18 06:00", "2014-09-18 12:00",
        "2014-09-18 18:00", "2014-09-19 00:00", "2014-09-19 06:00", "2014-09-19 12:00",
        "2014-09-19 18:00", "2014-09-20 00:00", "2014-09-20 06:00", "2014-09-20 12:00",
        "2014-09-20 18:00"
    ],
    "lat": [
        13.7, 14.2, 14.9, 15.8, 16.5, 17.2, 17.7, 18.3, 18.9, 19.6, 20.4, 21.4, 22.3,
        23.0, 23.7, 24.4, 25.1, 25.8, 26.5, 27.0, 27.7, 28.5, 29.5, 30.6, 31.7,
        32.9, 34.3, 35.7, 37.2, 38.3, 39.3, 39.9, 39.9, 39.9, 39.9, 39.5, 39.5,
        39.4, 39.3, 39.1, 38.8
    ],
    "lon": [
        -33.2, -34.4, -35.5, -36.5, -37.7, -39.2, -40.8, -42.3, -43.6, -44.8,
        -45.7, -46.4, -47.0, -47.9, -49.0, -50.3, -51.5, -52.7, -54.0, -55.1,
        -56.1, -56.7, -57.3, -57.8, -57.7, -57.0, -56.0, -54.5, -52.4, -49.5,
        -46.7, -44.1, -42.1, -40.1, -38.9, -38.1, -37.4, -36.6, -35.7, -34.4, -33.0
    ],
    "pressure": [
        1009, 1009, 1009, 1008, 1007, 1006, 1005, 1000, 999, 998, 997, 996, 995,
        994, 991, 985, 980, 975, 970, 966, 962, 960, 956, 955, 958, 959, 958,
        957, 955, 958, 964, 969, 973, 980, 988, 994, 1001, 1004, 1006, 1008, 1010
    ],
    "wind": [
        25, 25, 25, 30, 30, 35, 35, 40, 40, 40, 45, 45, 50, 55, 60, 70, 75, 80,
        85, 90, 95, 95, 100, 105, 95, 85, 85, 80, 80, 75, 70, 70, 65, 60, 55,
        45, 40, 35, 30, 30, 25
    ],
    "stage": [
        "low", "low", "low", "tropical depression", "tropical depression",
        "tropical storm", "tropical storm", "tropical storm", "tropical storm",
        "tropical storm", "tropical storm", "tropical storm", "tropical storm",
        "tropical storm", "tropical storm", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "tropical storm", "tropical storm", "tropical storm", "low", "low",
        "low", "low", "low"
    ]
}


In [ ]:
# ============================================================
# --- RADIAL LIGHTNING HOVMOLLER for Hurricane Edouard (2014)
# --- Matches publication Figure 4
# --- 20-km radial bins, hourly flash density (per 10,000 km²)
# --- Scaled for WWLLN detection efficiency
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import glob

# ---------------- PARAMETERS ----------------
km_per_deg = 111.0
radial_bin_km = 20.0  # publication uses 20 km bins
radial_bin_deg = radial_bin_km / km_per_deg
max_radius_km = 333.0
max_radius_deg = max_radius_km / km_per_deg
time_start = pd.Timestamp("2014-09-11 12:00")
time_end   = pd.Timestamp("2014-09-18 12:00")

# Detection efficiency (WWLLN ~5%)
detection_efficiency = 0.05

# ---------------- STORM TRACK ----------------
df_track = pd.DataFrame(track_data)
df_track["datetime"] = pd.to_datetime(df_track["datetime"])
df_track = df_track.set_index("datetime")

# Interpolate hourly (1-hr resolution)
interp_times = pd.date_range(start=df_track.index.min(),
                             end=df_track.index.max(),
                             freq="1H")

numeric_cols = ["lat", "lon", "wind", "pressure"]
track_interp = df_track[numeric_cols].reindex(df_track.index.union(interp_times))
track_interp = track_interp.interpolate(method="time")
track_interp = track_interp.loc[
    (track_interp.index >= time_start) &
    (track_interp.index <= time_end)
]
track_interp["stage"] = df_track["stage"].reindex(track_interp.index).ffill()

time_series = track_interp.index

# ---------------- LIGHTNING DATA ----------------
lightning_files = sorted(
    glob.glob("/nfs/home11/ugrad/2024/es348865/honors_research/250523/WWLLN_201409*.csv")
)
df_light_all = pd.concat([pd.read_csv(f) for f in lightning_files])
df_light_all["longitude"] = ((df_light_all["longitude"] + 180) % 360) - 180
df_light_all["datetime"] = pd.to_datetime(
    df_light_all["datetime"], format="%Y/%m/%d %H:%M:%S.%f"
)

# ---------------- RADIAL BINS ----------------
r_edges = np.arange(0, max_radius_deg + radial_bin_deg, radial_bin_deg)
r_centers = (r_edges[:-1] + r_edges[1:]) / 2

# Container: rows=time, cols=radius bins
LT = np.zeros((len(time_series), len(r_centers)))

# ---------------- BIN LIGHTNING BY RADIUS ----------------
for ti, t in enumerate(time_series):
    slat = track_interp.loc[t, "lat"]
    slon = track_interp.loc[t, "lon"]

    # ±30 min window (~1 hr)
    t0, t1 = t - pd.Timedelta(minutes=30), t + pd.Timedelta(minutes=30)
    df_sub = df_light_all[
        (df_light_all["datetime"] >= t0) &
        (df_light_all["datetime"] <= t1)
    ]

    if df_sub.empty:
        continue

    dx = (df_sub["longitude"].values - slon) * np.cos(np.deg2rad(slat))
    dy = df_sub["latitude"].values - slat
    r = np.sqrt(dx**2 + dy**2)

    mask = r <= max_radius_deg
    r = r[mask]
    if len(r) == 0:
        continue

    bin_ids = np.digitize(r, r_edges) - 1
    for bi in bin_ids:
        if 0 <= bi < len(r_centers):
            LT[ti, bi] += 1

# ---------------- FLASH DENSITY (per 10,000 km² per hour) ----------------
flash_density = np.zeros_like(LT, dtype=float)
for j in range(len(r_centers)):
    r1 = r_edges[j] * km_per_deg
    r2 = r_edges[j+1] * km_per_deg
    area = np.pi * (r2**2 - r1**2)  # area of annular ring in km²
    flash_density[:, j] = LT[:, j] / area * 1e4  # per 10,000 km²

# ---------------- SCALE FOR DETECTION EFFICIENCY ----------------
flash_density_corrected = flash_density / detection_efficiency

# Mask values <20 for shading (publication style)
flash_plot = np.ma.masked_less(flash_density_corrected, 20)

# ---------------- PLOTTING ----------------
fig, ax = plt.subplots(figsize=(12,6))

# Shading levels & colors (light → medium → dark → black)
shade_levels = [20, 80, 160, 320, 1e6]
shade_colors = ["#d9d9d9", "#969696", "#525252", "#000000"]

pcm = ax.contourf(
    mdates.date2num(time_series),
    r_centers * km_per_deg,
    flash_plot.T,
    levels=shade_levels,
    colors=shade_colors,
    extend='max'
)

# Contours at powers of 2
contour_levels = [20, 40, 80, 160, 320, 640]
cs = ax.contour(
    mdates.date2num(time_series),
    r_centers * km_per_deg,
    flash_plot.T,
    levels=contour_levels,
    colors='white',
    linewidths=0.8
)
ax.clabel(cs, fmt='%d', inline=True, fontsize=8, colors='white')

# Axis labels & title
ax.set_ylabel("Radius from center (km)")
ax.set_xlabel("Time")
ax.set_ylim(0, max_radius_km)
ax.set_title(
    "Lightning Radius–Time Hovmöller — Hurricane Edouard (2014)",
    y=1.12
)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))

# Daily separators
for d in pd.date_range(time_series[0].normalize(),
                       time_series[-1].normalize(),
                       freq="D"):
    ax.axvline(mdates.date2num(d), color="white", lw=0.6, alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# --- RADIAL LIGHTNING HOVMOLLER for Hurricane Edouard (2014)
# --- 20-km radial bins, hourly flash density (per 10,000 km²)
# --- Scaled for WWLLN detection efficiency
# --- Y-axis formatted like shear-relative azimuth
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import glob

# ---------------- PARAMETERS ----------------
km_per_deg = 111.0
radial_bin_km = 20.0
radial_bin_deg = radial_bin_km / km_per_deg
max_radius_km = 333.0
max_radius_deg = max_radius_km / km_per_deg
time_start = pd.Timestamp("2014-09-11 12:00")
time_end   = pd.Timestamp("2014-09-18 12:00")

detection_efficiency = 0.05  # scale WWLLN

# ---------------- STORM TRACK ----------------
df_track = pd.DataFrame(track_data)
df_track["datetime"] = pd.to_datetime(df_track["datetime"])
df_track = df_track.set_index("datetime")

interp_times = pd.date_range(start=df_track.index.min(),
                             end=df_track.index.max(),
                             freq="1H")

numeric_cols = ["lat", "lon", "wind", "pressure"]
track_interp = df_track[numeric_cols].reindex(df_track.index.union(interp_times))
track_interp = track_interp.interpolate(method="time")
track_interp = track_interp.loc[
    (track_interp.index >= time_start) &
    (track_interp.index <= time_end)
]
track_interp["stage"] = df_track["stage"].reindex(track_interp.index).ffill()
time_series = track_interp.index

# ---------------- LIGHTNING DATA ----------------
lightning_files = sorted(
    glob.glob("/nfs/home11/ugrad/2024/es348865/honors_research/250523/WWLLN_201409*.csv")
)
df_light_all = pd.concat([pd.read_csv(f) for f in lightning_files])
df_light_all["longitude"] = ((df_light_all["longitude"] + 180) % 360) - 180
df_light_all["datetime"] = pd.to_datetime(
    df_light_all["datetime"], format="%Y/%m/%d %H:%M:%S.%f"
)

# ---------------- RADIAL BINS ----------------
r_edges = np.arange(0, max_radius_deg + radial_bin_deg, radial_bin_deg)
r_centers = (r_edges[:-1] + r_edges[1:]) / 2
LT = np.zeros((len(time_series), len(r_centers)))

# ---------------- BIN LIGHTNING ----------------
for ti, t in enumerate(time_series):
    slat = track_interp.loc[t, "lat"]
    slon = track_interp.loc[t, "lon"]

    t0, t1 = t - pd.Timedelta(minutes=30), t + pd.Timedelta(minutes=30)
    df_sub = df_light_all[
        (df_light_all["datetime"] >= t0) &
        (df_light_all["datetime"] <= t1)
    ]
    if df_sub.empty:
        continue

    dx = (df_sub["longitude"].values - slon) * np.cos(np.deg2rad(slat))
    dy = df_sub["latitude"].values - slat
    r = np.sqrt(dx**2 + dy**2)
    mask = r <= max_radius_deg
    r = r[mask]
    if len(r) == 0:
        continue

    bin_ids = np.digitize(r, r_edges) - 1
    for bi in bin_ids:
        if 0 <= bi < len(r_centers):
            LT[ti, bi] += 1

# ---------------- FLASH DENSITY ----------------
flash_density = np.zeros_like(LT, dtype=float)
for j in range(len(r_centers)):
    r1 = r_edges[j] * km_per_deg
    r2 = r_edges[j+1] * km_per_deg
    area = np.pi * (r2**2 - r1**2)
    flash_density[:, j] = LT[:, j] / area * 1e4

# Scale for detection efficiency
flash_density_corrected = flash_density / detection_efficiency
flash_plot = np.ma.masked_less(flash_density_corrected, 20)

# ---------------- PLOTTING ----------------
fig, ax = plt.subplots(figsize=(12,6))

# Shading
shade_levels = [20, 80, 160, 320, 1e6]
shade_colors = ["#d9d9d9", "#969696", "#525252", "#000000"]

pcm = ax.contourf(
    mdates.date2num(time_series),
    r_centers * km_per_deg,
    flash_plot.T,
    levels=shade_levels,
    colors=shade_colors,
    extend='max'
)

# Contours at powers of 2
contour_levels = [20, 40, 80, 160, 320, 640]
ax.contour(
    mdates.date2num(time_series),
    r_centers * km_per_deg,
    flash_plot.T,
    levels=contour_levels,
    colors='white',
    linewidths=0.8
)

# ---------------- AXIS FORMATTING ----------------
# Invert Y axis like your shear-relative azimuth plot
ax.set_ylim(r_centers[-1] * km_per_deg, r_centers[0] * km_per_deg)  # max → min

# Remove labels and ticks for clean look
ax.set_ylabel("")
ax.set_xlabel("")
ax.set_xticks([])
ax.set_yticks([])

# X-axis limits
ax.set_xlim(time_start, time_end)

# Daily separators (still useful visually)
for day in pd.date_range(time_start.normalize(), time_end.normalize() + pd.Timedelta(days=1), freq="D"):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# --- RADIAL LIGHTNING HOVMOLLER for Hurricane Edouard (2014)
# --- 20-km radial bins, hourly flash density (per 10,000 km²)
# --- Scaled for WWLLN detection efficiency
# --- Clean axes, inverted y-axis
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import glob

# ---------------- PARAMETERS ----------------
km_per_deg = 111.0
radial_bin_km = 20.0
radial_bin_deg = radial_bin_km / km_per_deg
max_radius_km = 333.0
max_radius_deg = max_radius_km / km_per_deg
time_start = pd.Timestamp("2014-09-11 12:00")
time_end   = pd.Timestamp("2014-09-18 12:00")
detection_efficiency = 0.05  # scale WWLLN data

# ---------------- STORM TRACK ----------------
df_track = pd.DataFrame(track_data)
df_track["datetime"] = pd.to_datetime(df_track["datetime"])
df_track = df_track.set_index("datetime")

interp_times = pd.date_range(start=df_track.index.min(),
                             end=df_track.index.max(),
                             freq="1H")

numeric_cols = ["lat", "lon", "wind", "pressure"]
track_interp = df_track[numeric_cols].reindex(df_track.index.union(interp_times))
track_interp = track_interp.interpolate(method="time")
track_interp = track_interp.loc[
    (track_interp.index >= time_start) &
    (track_interp.index <= time_end)
]
track_interp["stage"] = df_track["stage"].reindex(track_interp.index).ffill()
time_series = track_interp.index

# ---------------- LIGHTNING DATA ----------------
lightning_files = sorted(
    glob.glob("/nfs/home11/ugrad/2024/es348865/honors_research/250523/WWLLN_201409*.csv")
)
df_light_all = pd.concat([pd.read_csv(f) for f in lightning_files])
df_light_all["longitude"] = ((df_light_all["longitude"] + 180) % 360) - 180
df_light_all["datetime"] = pd.to_datetime(
    df_light_all["datetime"], format="%Y/%m/%d %H:%M:%S.%f"
)

# ---------------- RADIAL BINS ----------------
r_edges = np.arange(0, max_radius_deg + radial_bin_deg, radial_bin_deg)
r_centers = (r_edges[:-1] + r_edges[1:]) / 2
LT = np.zeros((len(time_series), len(r_centers)))

# ---------------- BIN LIGHTNING ----------------
for ti, t in enumerate(time_series):
    slat = track_interp.loc[t, "lat"]
    slon = track_interp.loc[t, "lon"]

    t0, t1 = t - pd.Timedelta(minutes=30), t + pd.Timedelta(minutes=30)
    df_sub = df_light_all[
        (df_light_all["datetime"] >= t0) &
        (df_light_all["datetime"] <= t1)
    ]
    if df_sub.empty:
        continue

    dx = (df_sub["longitude"].values - slon) * np.cos(np.deg2rad(slat))
    dy = df_sub["latitude"].values - slat
    r = np.sqrt(dx**2 + dy**2)
    mask = r <= max_radius_deg
    r = r[mask]
    if len(r) == 0:
        continue

    bin_ids = np.digitize(r, r_edges) - 1
    for bi in bin_ids:
        if 0 <= bi < len(r_centers):
            LT[ti, bi] += 1

# ---------------- FLASH DENSITY ----------------
flash_density = np.zeros_like(LT, dtype=float)
for j in range(len(r_centers)):
    r1 = r_edges[j] * km_per_deg
    r2 = r_edges[j+1] * km_per_deg
    area = np.pi * (r2**2 - r1**2)
    flash_density[:, j] = LT[:, j] / area * 1e4  # per 10,000 km²

# Scale for detection efficiency
flash_density_corrected = flash_density / detection_efficiency
flash_plot = np.ma.masked_less(flash_density_corrected, 20)

# ---------------- PLOTTING ----------------
fig, ax = plt.subplots(figsize=(12,6))

# Shading
shade_levels = [20, 80, 160, 320, 1e6]
shade_colors = ["#d9d9d9", "#969696", "#525252", "#000000"]

pcm = ax.contourf(
    mdates.date2num(time_series),
    r_centers * km_per_deg,
    flash_plot.T,
    levels=shade_levels,
    colors=shade_colors,
    extend='max'
)

# Contours at powers of 2
contour_levels = [20, 40, 80, 160, 320, 640]
ax.contour(
    mdates.date2num(time_series),
    r_centers * km_per_deg,
    flash_plot.T,
    levels=contour_levels,
    colors='white',
    linewidths=0.8
)

# ---------------- AXIS FORMATTING ----------------
# ---------------- AXIS FORMATTING ----------------
# Invert Y-axis
ax.set_ylim(r_centers[-1] * km_per_deg, r_centers[0] * km_per_deg)

# Keep tick marks visible, remove numeric labels
ax.tick_params(axis='both', which='both', labelleft=True, labelbottom=True, length=5)

# Make sure spines are visible
for spine in ax.spines.values():
    spine.set_visible(True)

# X-axis limits
ax.set_xlim(time_start, time_end)

# Daily separators
for day in pd.date_range(time_start.normalize(), time_end.normalize() + pd.Timedelta(days=1), freq="D"):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter

fig, ax = plt.subplots(figsize=(12,6))

# ---------------- SHADING ----------------
shade_levels = [20, 80, 160, 320, 1e6]
shade_colors = ["#d9d9d9", "#969696", "#525252", "#000000"]

pcm = ax.contourf(
    mdates.date2num(time_series),
    r_centers * km_per_deg,
    flash_plot.T,
    levels=shade_levels,
    colors=shade_colors,
    extend='max'
)

# Contours at powers of 2
contour_levels = [20, 40, 80, 160, 320, 640]
ax.contour(
    mdates.date2num(time_series),
    r_centers * km_per_deg,
    flash_plot.T,
    levels=contour_levels,
    colors='white',
    linewidths=0.8
)

# ---------------- AXES FORMATTING ----------------
# ---------------- AXES FORMATTING ----------------
ax.set_ylabel("Radius from center (km)", labelpad=1)

# Custom y-axis positions for 90 -> 0 -> 270 -> 180 -> 90
# Map these angular-like positions to your radial bins
# We'll use r_centers scaled to km
y_labels = [90, 0, 270, 180, 90]

# Choose radial bins that match the positions (for simplicity, evenly spaced along r_centers)
yticks_pos = np.linspace(r_centers[0]*km_per_deg, r_centers[-1]*km_per_deg, num=len(y_labels))

ax.set_yticks(yticks_pos)
ax.set_yticklabels(y_labels)

# Invert y-axis so 90 starts at the bottom
#ax.invert_yaxis()


ax.set_xlim(time_start, time_end)
ax.set_title(
    f"Lightning Radius–Time Hovmöller\n{time_start:%Y-%m-%d %H:%M} – {time_end:%Y-%m-%d %H:%M}",
    y=1.15
)
ax.xaxis_date()
ax.xaxis.set_major_formatter(DateFormatter('%m-%d\n%H:%M'))

# Daily separators
for day in pd.date_range(time_start.normalize(), time_end.normalize() + pd.Timedelta(days=1), freq="D"):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.6)


plt.tight_layout()
fig.savefig("lightning_radius_time_hovmoller.png", dpi=300, bbox_inches="tight")
plt.show()


